In [2]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles


ModuleNotFoundError: No module named 'cv2'

In [ ]:
from IPython.display import display

IMAGE_DIR = Path("/Users/macbook/Documents/EduVisionSeat/EduVisionSeat/data/images/input")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

image_paths = [path for path in IMAGE_DIR.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS]

if not image_paths:
    raise FileNotFoundError(f"No images were found in {IMAGE_DIR}. Add image files there and rerun this cell.")

DESIRED_HEIGHT = 480
DESIRED_WIDTH = 480


def resize_and_show(image, title=None):
    h, w = image.shape[:2]
    if h < w:
        img = cv2.resize(image, (DESIRED_WIDTH, math.floor(h / (w / DESIRED_WIDTH))))
    else:
        img = cv2.resize(image, (math.floor(w / (h / DESIRED_HEIGHT)), DESIRED_HEIGHT))
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


images = {}
for path in image_paths:
    image = cv2.imread(str(path))
    if image is not None:
        images[path.name] = image

if not images:
    raise RuntimeError("No valid images could be loaded from the image directory.")

for name, image in images.items():
    print(name)
    resize_and_show(image, title=name)


In [ ]:
help(mp_pose.Pose)


In [ ]:
# Run MediaPipe Pose and draw pose landmarks.
with mp_pose.Pose(
    static_image_mode=True, min_detection_confidence=0.5, model_complexity=2
) as pose:
    for name, image in images.items():
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        image_height, image_width, _ = image.shape
        if not results.pose_landmarks:
            print(f"No pose landmarks detected for {name}")
            continue

        print(
            f"Nose coordinates: ({results.pose_landmarks.landmark[mp_pose.PoseLandmark.NOSE].x * image_width}, "
            f"{results.pose_landmarks.landmark[mp_pose.PoseLandmark.NOSE].y * image_height})"
        )

        annotated_image = image.copy()
        mp_drawing.draw_landmarks(
            annotated_image,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style(),
        )
        resize_and_show(annotated_image, title=f"Pose landmarks - {name}")


In [ ]:
# Run MediaPipe Pose and plot 3D pose world landmarks.
with mp_pose.Pose(
    static_image_mode=True, min_detection_confidence=0.5, model_complexity=2
) as pose:
    for name, image in images.items():
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.pose_world_landmarks is None:
            print(f"No world landmarks available for {name}")
            continue

        print(f"Nose world landmark for {name}:")
        print(results.pose_world_landmarks.landmark[mp_pose.PoseLandmark.NOSE])

        mp_drawing.plot_landmarks(results.pose_world_landmarks, mp_pose.POSE_CONNECTIONS)


In [ ]:
# Run MediaPipe Pose with `enable_segmentation=True` to get pose segmentation.
with mp_pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.5,
    model_complexity=2,
    enable_segmentation=True,
) as pose:
    for name, image in images.items():
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.segmentation_mask is None:
            print(f"Segmentation mask unavailable for {name}")
            continue

        annotated_image = image.copy()
        red_img = np.zeros_like(annotated_image, dtype=np.uint8)
        red_img[:, :] = (255, 255, 255)
        segm_2class = 0.2 + 0.8 * results.segmentation_mask
        segm_2class = np.repeat(segm_2class[..., np.newaxis], 3, axis=2)
        annotated_image = annotated_image * segm_2class + red_img * (1 - segm_2class)
        resize_and_show(annotated_image, title=f"Pose segmentation - {name}")
